# Reconhecimento de Gestos com Desenho Completo das Mãos

Este notebook utiliza a **MediaPipe Tasks API** para reconhecer gestos e desenhar todas as conexões (esqueleto) das mãos.

### Controles:
- **'s'**: Salvar captura atual.
- **'q'**: Sair.

In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import os
import requests
from datetime import datetime

In [2]:
output_folder = "capturas_gestos"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

model_path = 'gesture_recognizer.task'
url = "https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task"

if not os.path.exists(model_path):
    print("Baixando modelo...")
    r = requests.get(url)
    with open(model_path, 'wb') as f: f.write(r.content)
    print("Download concluído!")

In [3]:
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.GestureRecognizerOptions(
    base_options=base_options,
    num_hands=2
)
recognizer = vision.GestureRecognizer.create_from_options(options)

In [4]:
def draw_landmarks_on_image(rgb_image, detection_result):
    hand_landmarks_list = detection_result.hand_landmarks
    gestures_list = detection_result.gestures
    annotated_image = np.copy(rgb_image)

    # Conexões da mão (MediaPipe Hand Connections)
    HAND_CONNECTIONS = [
        (0, 1), (1, 2), (2, 3), (3, 4),           # Polegar
        (0, 5), (5, 6), (6, 7), (7, 8),           # Indicador
        (5, 9), (9, 10), (10, 11), (11, 12),      # Médio
        (9, 13), (13, 14), (14, 15), (15, 16),    # Anelar
        (13, 17), (17, 18), (18, 19), (19, 20),   # Mínimo
        (0, 17)                                   # Palma
    ]

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        gesture = gestures_list[idx][0]
        
        # 1. Desenhar Conexões
        for connection in HAND_CONNECTIONS:
            start_idx = connection[0]
            end_idx = connection[1]
            
            x1 = int(hand_landmarks[start_idx].x * annotated_image.shape[1])
            y1 = int(hand_landmarks[start_idx].y * annotated_image.shape[0])
            x2 = int(hand_landmarks[end_idx].x * annotated_image.shape[1])
            y2 = int(hand_landmarks[end_idx].y * annotated_image.shape[0])
            
            cv2.line(annotated_image, (x1, y1), (x2, y2), (255, 255, 255), 2)

        # 2. Desenhar Pontos (Landmarks)
        for landmark in hand_landmarks:
            x = int(landmark.x * annotated_image.shape[1])
            y = int(landmark.y * annotated_image.shape[0])
            # Cores diferentes para articulações
            cv2.circle(annotated_image, (x, y), 5, (0, 255, 0), -1)
            cv2.circle(annotated_image, (x, y), 2, (0, 0, 255), -1)

        # 3. Escrever o Gesto
        tx = int(hand_landmarks[0].x * annotated_image.shape[1])
        ty = int(hand_landmarks[0].y * annotated_image.shape[0])
        label = f"{gesture.category_name} ({gesture.score:.2f})"
        cv2.putText(annotated_image, label, (tx, ty - 20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

    return annotated_image

In [5]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    frame = cv2.flip(frame, 1)
    rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    result = recognizer.recognize(mp_image)
    
    # Desenhar esqueleto e pontos se houver detecção
    if result.hand_landmarks:
        display_frame = draw_landmarks_on_image(frame, result)
    else:
        display_frame = frame.copy()

    cv2.imshow('MediaPipe Hand Landmarks Rendering', display_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{output_folder}/esqueleto_mao_{timestamp}.jpg"
        cv2.imwrite(filename, display_frame)
        print(f"Salvo: {filename}")
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Salvo: capturas_gestos/esqueleto_mao_20260316_001054.jpg
Salvo: capturas_gestos/esqueleto_mao_20260316_001056.jpg
Salvo: capturas_gestos/esqueleto_mao_20260316_001102.jpg
Salvo: capturas_gestos/esqueleto_mao_20260316_001109.jpg
Salvo: capturas_gestos/esqueleto_mao_20260316_001115.jpg
Salvo: capturas_gestos/esqueleto_mao_20260316_001122.jpg
Salvo: capturas_gestos/esqueleto_mao_20260316_001125.jpg
